# Introduction à PySpark – Traitement Distribué d'Images

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tanouar/1to1code/blob/main/pySpark/notebooks/04_traitement_images.ipynb)

Notebook conçu pour **Google Colab**.  
Dataset : **Oxford-IIIT Pet Dataset** (~7 300 images de chats et chiens).  
On explore le chargement `binaryFile`, les UDF Pillow, et l'impact du cache.

> **SparkUI** : accessible après la cellule de setup via le lien affiché.


In [1]:
# Installation + setup (Colab)
!pip install -q pyspark findspark
!git clone --filter=blob:none --sparse https://github.com/tanouar/1to1code.git -q \
  && cd 1to1code && git sparse-checkout set pySpark -q

import os, sys
PYSPARK_TOOLS = os.path.join('/content', '1to1code', 'pySpark')
if PYSPARK_TOOLS not in sys.path:
    sys.path.insert(0, PYSPARK_TOOLS)

from sparktools import setup_colab, download_oxford_pets
from sparktools.image_utils import build_metadata_pipeline, add_image_dimensions
from pyspark.sql import functions as F

spark, monitor, helper = setup_colab("PySpark - Traitement Images")


fatal: destination path '1to1code' already exists and is not an empty directory.
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

✓ SparkUI ouverte → cliquez sur le lien ci-dessus
✓ Spark 4.0.2  |  App : PySpark - Traitement Images
✓ Memory : 4g  |  Shuffle partitions : 8
✓ SparkJobMonitor et SparkHelper initialisés


## 1. Téléchargement du Dataset Oxford Pets

`download_oxford_pets()` télécharge (~800 MB) et extrait le dataset dans `/content/oxford_pets/`.


In [2]:
images_path = download_oxford_pets()   # télécharge + extrait dans /content/oxford_pets/images/
print(f"Images prêtes : {images_path}")


⏳ Téléchargement Oxford Pets (~800 MB) — 5 à 10 minutes...
  100 / 755 MB  (13%)
  200 / 755 MB  (26%)
  300 / 755 MB  (40%)
  400 / 755 MB  (53%)
  500 / 755 MB  (66%)
  600 / 755 MB  (79%)
  700 / 755 MB  (93%)
⏳ Extraction de l'archive...
✓ Oxford Pets prêt → /content/oxford_pets/images  (7,390 images)
Images prêtes : /content/oxford_pets/images


## 2. Chargement Distribué – binaryFile

`spark.read.format("binaryFile")` charge les images en **parallèle**.  
Colonnes automatiques : `path`, `modificationTime`, `length`, `content` (bytes bruts).  
→ SparkUI : toujours lazy — aucun job après cette cellule.


In [3]:
import time

start = time.time()
df_images = spark.read.format("binaryFile") \
    .option("pathGlobFilter", "*.jpg") \
    .load(images_path)
elapsed = time.time() - start

df_images.printSchema()
print(f"\nPlan créé en {elapsed:.4f}s — aucune image lue (lazy).")
print("→ SparkUI : aucun job ne doit apparaître.")


root
 |-- path: string (nullable = true)
 |-- modificationTime: timestamp (nullable = true)
 |-- length: long (nullable = true)
 |-- content: binary (nullable = true)


Plan créé en 3.3161s — aucune image lue (lazy).
→ SparkUI : aucun job ne doit apparaître.


## 3. Première Action – count()

→ SparkUI : un job apparaît. Spark lit les métadonnées des fichiers.


In [4]:
result = monitor.execute_and_monitor(
    lambda: df_images.count(),
    "JOB 1: Compter les images"
)
print(f"\n{result:,} images")

monitor.execute_and_monitor(
    lambda: df_images.select("path", "length", "modificationTime").show(5, truncate=False),
    "JOB 2: Afficher un échantillon"
)



🔵 JOB 1: Compter les images
📌 Tracking ID: 809e4914

✅ SUCCESS
⏱️  Durée: 10.16s
📊 Spark Job ID(s): [1, 0]
📦 Stages: 3 | Tasks: 469
🌐 Spark UI: http://localhost:4040/jobs/

7,390 images

🔵 JOB 2: Afficher un échantillon
📌 Tracking ID: 1e084c78
+-----------------------------------------------------+-------+-------------------+
|path                                                 |length |modificationTime   |
+-----------------------------------------------------+-------+-------------------+
|file:/content/oxford_pets/images/Egyptian_Mau_14.jpg |1125149|2012-06-18 16:52:58|
|file:/content/oxford_pets/images/Egyptian_Mau_165.jpg|1061923|2012-06-18 16:53:21|
|file:/content/oxford_pets/images/Egyptian_Mau_162.jpg|968009 |2012-06-18 16:54:01|
|file:/content/oxford_pets/images/Egyptian_Mau_196.jpg|661645 |2012-06-18 16:54:12|
|file:/content/oxford_pets/images/Abyssinian_178.jpg  |621103 |2012-06-18 16:54:21|
+-----------------------------------------------------+-------+-------------------+

## 4. Extraction des Métadonnées

`build_metadata_pipeline()` est une transformation **lazy** qui extrait depuis le nom de fichier :  
`filename`, `race`, `image_id`, `species` (chat vs chien), `size_mb`.


In [5]:
# Transformation lazy : extraction des métadonnées depuis les noms de fichiers
df_meta = build_metadata_pipeline(df_images)

# Action : afficher
monitor.execute_and_monitor(
    lambda: df_meta.select("filename", "race", "species", "image_id", "size_mb")
        .show(10, truncate=False),
    "JOB 3: Afficher métadonnées"
)



🔵 JOB 3: Afficher métadonnées
📌 Tracking ID: 8762ac51
+--------------------------+------------------+-------+--------+-------+
|filename                  |race              |species|image_id|size_mb|
+--------------------------+------------------+-------+--------+-------+
|Egyptian_Mau_14.jpg       |Egyptian Mau      |Cat    |14      |1.07   |
|Egyptian_Mau_165.jpg      |Egyptian Mau      |Cat    |165     |1.01   |
|Egyptian_Mau_162.jpg      |Egyptian Mau      |Cat    |162     |0.92   |
|Egyptian_Mau_196.jpg      |Egyptian Mau      |Cat    |196     |0.63   |
|Abyssinian_178.jpg        |Abyssinian        |Cat    |178     |0.59   |
|Bombay_29.jpg             |Bombay            |Cat    |29      |0.49   |
|Egyptian_Mau_182.jpg      |Egyptian Mau      |Cat    |182     |0.44   |
|Abyssinian_170.jpg        |Abyssinian        |Cat    |170     |0.43   |
|Abyssinian_40.jpg         |Abyssinian        |Cat    |40      |0.35   |
|german_shorthaired_119.jpg|german shorthaired|Dog    |119     |0.34 

## 5. Statistiques sur le Dataset

→ SparkUI : observez le plan SQL dans **SQL / DataFrame** pour chaque job.


In [6]:
monitor.execute_and_monitor(
    lambda: df_meta.groupBy("species")
        .agg(
            F.count("*").alias("nb_images"),
            F.avg("size_mb").alias("taille_moyenne_mb"),
            F.min("size_mb").alias("taille_min_mb"),
            F.max("size_mb").alias("taille_max_mb")
        )
        .show(truncate=False),
    "JOB 4: Stats par espèce"
)

monitor.execute_and_monitor(
    lambda: df_meta.groupBy("race", "species")
        .agg(F.count("*").alias("nb_images"))
        .orderBy(F.col("nb_images").desc())
        .limit(10)
        .show(truncate=False),
    "JOB 5: Top 10 races"
)



🔵 JOB 4: Stats par espèce
📌 Tracking ID: f31ae9e2
+-------+---------+-----------------+-------------+-------------+
|species|nb_images|taille_moyenne_mb|taille_min_mb|taille_max_mb|
+-------+---------+-----------------+-------------+-------------+
|Cat    |2400     |0.086488         |0.00         |1.07         |
|Dog    |4990     |0.109900         |0.00         |0.34         |
+-------+---------+-----------------+-------------+-------------+


✅ SUCCESS
⏱️  Durée: 9.81s
📊 Spark Job ID(s): [5, 4]
📦 Stages: 3 | Tasks: 469
🌐 Spark UI: http://localhost:4040/jobs/

🔵 JOB 5: Top 10 races
📌 Tracking ID: 4e5706b9
+----------------------+-------+---------+
|race                  |species|nb_images|
+----------------------+-------+---------+
|Bombay                |Cat    |200      |
|german shorthaired    |Dog    |200      |
|english cocker spaniel|Dog    |200      |
|keeshond              |Dog    |200      |
|Bengal                |Cat    |200      |
|leonberger            |Dog    |200      |

## 6. Dimensions des Images – UDF Pillow

`add_image_dimensions()` applique une **UDF** qui décode chaque image avec Pillow  
et extrait : `width`, `height`, `channels`, `format`, `pixels`, `aspect_ratio`.  
→ SparkUI : le job sera long — chaque Task décode une image.


In [7]:
# Transformation lazy : décodage des bytes avec l'UDF Pillow
df_dims = add_image_dimensions(df_meta)

print("UDF ajoutée au plan (lazy).")
print("→ SparkUI : aucun job encore.")

# Action : décoder toutes les images
monitor.execute_and_monitor(
    lambda: df_dims.select("filename", "race", "width", "height", "channels", "aspect_ratio")
        .show(10, truncate=False),
    "JOB 6: Décoder les images – UDF Pillow"
)


UDF ajoutée au plan (lazy).
→ SparkUI : aucun job encore.

🔵 JOB 6: Décoder les images – UDF Pillow
📌 Tracking ID: ca82842e
+--------------------------+------------------+-----+------+--------+------------+
|filename                  |race              |width|height|channels|aspect_ratio|
+--------------------------+------------------+-----+------+--------+------------+
|Egyptian_Mau_14.jpg       |Egyptian Mau      |582  |800   |4       |0.73        |
|Egyptian_Mau_165.jpg      |Egyptian Mau      |2560 |1920  |3       |1.33        |
|Egyptian_Mau_162.jpg      |Egyptian Mau      |3264 |2448  |3       |1.33        |
|Egyptian_Mau_196.jpg      |Egyptian Mau      |1886 |2606  |3       |0.72        |
|Abyssinian_178.jpg        |Abyssinian        |400  |400   |3       |1.00        |
|Bombay_29.jpg             |Bombay            |1600 |1200  |3       |1.33        |
|Egyptian_Mau_182.jpg      |Egyptian Mau      |1999 |1499  |3       |1.33        |
|Abyssinian_170.jpg        |Abyssinian        

## 7. Statistiques des Dimensions


In [8]:
monitor.execute_and_monitor(
    lambda: df_dims.select("width", "height", "channels").describe().show(),
    "JOB 7: Stats dimensions"
)

monitor.execute_and_monitor(
    lambda: df_dims.withColumn(
        "orientation",
        F.when(F.col("width") == F.col("height"), "Carré")
         .when(F.col("width") > F.col("height"), "Paysage")
         .otherwise("Portrait")
    ).groupBy("orientation").agg(F.count("*").alias("nb")).show(),
    "JOB 8: Distribution orientations"
)

monitor.show_history()



🔵 JOB 7: Stats dimensions
📌 Tracking ID: 74eeb34f
+-------+------------------+------------------+-------------------+
|summary|             width|            height|           channels|
+-------+------------------+------------------+-------------------+
|  count|              7390|              7390|               7390|
|   mean| 436.7451962110961| 390.9136671177267| 2.9979702300405955|
| stddev|115.87730011126128|109.58632393092991|0.07262228029252112|
|    min|               114|               103|                  1|
|    max|              3264|              2606|                  4|
+-------+------------------+------------------+-------------------+


✅ SUCCESS
⏱️  Durée: 112.37s
📊 Spark Job ID(s): [10, 9]
📦 Stages: 3 | Tasks: 469
🌐 Spark UI: http://localhost:4040/jobs/

🔵 JOB 8: Distribution orientations
📌 Tracking ID: df17c498
+-----------+----+
|orientation|  nb|
+-----------+----+
|      Carré| 116|
|    Paysage|4902|
|   Portrait|2372|
+-----------+----+


✅ SUCCESS
⏱️  Durée

## 8. Cache et Performance

→ SparkUI → **Storage** : observez les données en mémoire après `cache()`.  
→ Comparez les durées des 3 jobs dans l'onglet **Jobs**.


In [9]:
# Sans cache
result1 = monitor.execute_and_monitor(
    lambda: df_dims.filter(F.col("species") == "Cat").count(),
    "JOB: Count chats (sans cache)"
)
print(f"{result1:,} images de chats")

# Mise en cache
df_dims.cache()

# Avec cache (premier passage — populate)
monitor.execute_and_monitor(
    lambda: df_dims.filter(F.col("species") == "Cat").count(),
    "JOB: Count chats (mise en cache)"
)

# Depuis cache
monitor.execute_and_monitor(
    lambda: df_dims.filter(F.col("species") == "Cat").count(),
    "JOB: Count chats (depuis cache)"
)

print("→ SparkUI → Storage : observez les données en mémoire.")
print("→ Comparez les durées des 3 jobs dans l'onglet Jobs.")

df_dims.unpersist()



🔵 JOB: Count chats (sans cache)
📌 Tracking ID: 0cf35e41

✅ SUCCESS
⏱️  Durée: 2.96s
📊 Spark Job ID(s): [14, 13]
📦 Stages: 3 | Tasks: 469
🌐 Spark UI: http://localhost:4040/jobs/
2,400 images de chats

🔵 JOB: Count chats (mise en cache)
📌 Tracking ID: 5e10648f

✅ SUCCESS
⏱️  Durée: 116.42s
📊 Spark Job ID(s): [17, 16, 15]
📦 Stages: 4 | Tasks: 703
🌐 Spark UI: http://localhost:4040/jobs/

🔵 JOB: Count chats (depuis cache)
📌 Tracking ID: 91fe6de9

✅ SUCCESS
⏱️  Durée: 4.24s
📊 Spark Job ID(s): [19, 18]
📦 Stages: 3 | Tasks: 469
🌐 Spark UI: http://localhost:4040/jobs/
→ SparkUI → Storage : observez les données en mémoire.
→ Comparez les durées des 3 jobs dans l'onglet Jobs.


DataFrame[path: string, modificationTime: timestamp, length: bigint, content: binary, filename: string, race_raw: string, race: string, image_id: int, species: string, size_mb: decimal(10,2), width: int, height: int, channels: int, format: string, pixels: int, aspect_ratio: decimal(10,2)]

In [10]:
# ============================================
# PARTIE 9 : Redimensionnement Distribué
# ============================================

print("\n✂️ PARTIE 9 : Redimensionnement Massif d'Images")
print("="*70)

print("""
🧠 CONCEPT : Map Operation Distribuée
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• Redimensionner 7000+ images en parallèle
• Chaque worker traite une partie des images
• UDF appliquée de manière distribuée
• Gain de temps significatif vs traitement séquentiel
""")

# UDF pour redimensionner
@udf(returnType=BinaryType())
def resize_image(image_bytes, target_width=224, target_height=224):
    """Redimensionne une image à une taille fixe"""
    try:
        img = Image.open(io.BytesIO(image_bytes))

        # Redimensionner
        img_resized = img.resize((target_width, target_height), Image.LANCZOS)

        # Convertir en bytes
        buffer = io.BytesIO()
        img_resized.save(buffer, format='JPEG')

        return buffer.getvalue()
    except Exception as e:
        return None

print("\n⏱️ Redimensionnement de TOUTES les images à 224×224...")
print("   (Taille standard pour les modèles de deep learning)")

# On va travailler sur un échantillon pour aller plus vite
df_sample = df_with_dims.limit(1000)

result = monitor.execute_and_monitor(
    lambda: df_sample.withColumn(
        "resized_content",
        resize_image(F.col("content"), F.lit(224), F.lit(224))
    ).select("filename", "width", "height")
     .show(10),
    "JOB 16: Redimensionner 1000 images"
)

print("\n💡 OBSERVATIONS :")
print("   • Redimensionnement distribué sur tous les workers")
print("   • Chaque worker traite sa partition en parallèle")
print("   • Gain de temps vs traitement séquentiel")


✂️ PARTIE 9 : Redimensionnement Massif d'Images

🧠 CONCEPT : Map Operation Distribuée
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• Redimensionner 7000+ images en parallèle
• Chaque worker traite une partie des images
• UDF appliquée de manière distribuée
• Gain de temps significatif vs traitement séquentiel



NameError: name 'udf' is not defined

In [ ]:
# ============================================
# PARTIE 10 : Feature Extraction Simple
# ============================================

print("\n🎨 PARTIE 10 : Extraction de Features Simples")
print("="*70)

print("""
🧠 CONCEPT : Feature Engineering Distribué
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• Extraire des caractéristiques de chaque image
• Couleur dominante (moyenne RGB)
• Luminosité moyenne
• Tous les calculs en parallèle
""")

# UDF pour extraire les features
@udf(returnType=StructType([
    StructField("avg_red", FloatType(), True),
    StructField("avg_green", FloatType(), True),
    StructField("avg_blue", FloatType(), True),
    StructField("brightness", FloatType(), True)
]))
def extract_color_features(image_bytes):
    """Extrait les features de couleur"""
    try:
        img = Image.open(io.BytesIO(image_bytes))

        # Convertir en RGB si nécessaire
        if img.mode != 'RGB':
            img = img.convert('RGB')

        # Réduire la taille pour accélérer
        img_small = img.resize((50, 50))

        # Extraire les pixels
        pixels = list(img_small.getdata())

        # Calculer les moyennes
        avg_r = sum(p[0] for p in pixels) / len(pixels)
        avg_g = sum(p[1] for p in pixels) / len(pixels)
        avg_b = sum(p[2] for p in pixels) / len(pixels)

        # Luminosité (formule standard)
        brightness = (0.299 * avg_r + 0.587 * avg_g + 0.114 * avg_b)

        return {
            'avg_red': float(avg_r),
            'avg_green': float(avg_g),
            'avg_blue': float(avg_b),
            'brightness': float(brightness)
        }
    except Exception as e:
        return None

print("\n⏱️ Extraction des features de couleur...")

result = monitor.execute_and_monitor(
    lambda: df_sample.withColumn("color_features", extract_color_features(F.col("content")))
        .withColumn("avg_red", F.col("color_features.avg_red"))
        .withColumn("avg_green", F.col("color_features.avg_green"))
        .withColumn("avg_blue", F.col("color_features.avg_blue"))
        .withColumn("brightness", F.col("color_features.brightness"))
        .select("filename", "race", "avg_red", "avg_green", "avg_blue", "brightness")
        .show(10),
    "JOB 17: Extraction features couleur"
)

print("\n💡 Utilisations possibles :")
print("   • Filtrer les images trop sombres/claires")
print("   • Grouper par couleur dominante")
print("   • Détecter les images en noir et blanc")


🎨 PARTIE 10 : Extraction de Features Simples

🧠 CONCEPT : Feature Engineering Distribué
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• Extraire des caractéristiques de chaque image
• Couleur dominante (moyenne RGB)
• Luminosité moyenne
• Tous les calculs en parallèle


⏱️ Extraction des features de couleur...

🔵 JOB 17: Extraction features couleur
📌 Tracking ID: 93a3d89a
+--------------------+------------------+-------+---------+--------+----------+
|            filename|              race|avg_red|avg_green|avg_blue|brightness|
+--------------------+------------------+-------+---------+--------+----------+
| Egyptian_Mau_14.jpg|      Egyptian Mau|   NULL|     NULL|    NULL|      NULL|
|Egyptian_Mau_165.jpg|      Egyptian Mau|   NULL|     NULL|    NULL|      NULL|
|Egyptian_Mau_162.jpg|      Egyptian Mau|   NULL|     NULL|    NULL|      NULL|
|Egyptian_Mau_196.jpg|      Egyptian Mau|   NULL|     NULL|    NULL|      NULL|
|  Abyssinian_178.jpg|        Abyssinian|   NULL| 

In [ ]:
# ============================================
# PARTIE 11 : Partitionnement et Sauvegarde
# ============================================

print("\n💾 PARTIE 11 : Partitionnement et Sauvegarde")
print("="*70)

print("""
🧠 CONCEPT : Partitionnement par Catégorie
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• Organiser les images par espèce (Cat/Dog)
• Puis par race
• Permet la lecture sélective (partition pruning)
• Format Parquet pour stocker les données binaires
""")

output_path = "/content/oxford_pets_processed"

print("\n📝 Sauvegarde partitionnée par espèce et race...")
print("   (Sur un échantillon de 1000 images)")

# Préparer le DataFrame
df_to_save = df_sample.select(
    "filename",
    "species",
    "race",
    "width",
    "height",
    "channels",
    "size_mb",
    "content"
)

result = monitor.execute_and_monitor(
    lambda: df_to_save.write
        .mode("overwrite")
        .partitionBy("species", "race")
        .parquet(output_path),
    "JOB 18: Sauvegarde partitionnée"
)

print(f"\n✓ Données sauvegardées dans: {output_path}")

# Afficher la structure
print("\n📁 Structure des partitions:")
for root, dirs, files in os.walk(output_path):
    level = root.replace(output_path, '').count(os.sep)
    indent = '   ' * level
    folder = os.path.basename(root)
    if folder and level < 3:
        print(f"{indent}📂 {folder}/")

    if files and level < 3:
        parquet_files = [f for f in files if f.endswith('.parquet')]
        if parquet_files:
            print(f"{indent}   📄 {len(parquet_files)} fichier(s) .parquet")


💾 PARTIE 11 : Partitionnement et Sauvegarde

🧠 CONCEPT : Partitionnement par Catégorie
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• Organiser les images par espèce (Cat/Dog)
• Puis par race
• Permet la lecture sélective (partition pruning)
• Format Parquet pour stocker les données binaires


📝 Sauvegarde partitionnée par espèce et race...
   (Sur un échantillon de 1000 images)

🔵 JOB 18: Sauvegarde partitionnée
📌 Tracking ID: f45b96d7

✅ SUCCESS
⏱️  Durée: 30.57s
📊 Spark Job ID(s): [33, 32]

✓ Données sauvegardées dans: /content/oxford_pets_processed

📁 Structure des partitions:
📂 oxford_pets_processed/
   📂 species=Cat/
      📂 race=British Shorthair/
         📄 1 fichier(s) .parquet
      📂 race=Egyptian Mau/
         📄 1 fichier(s) .parquet
      📂 race=Russian Blue/
         📄 1 fichier(s) .parquet
      📂 race=Maine Coon/
         📄 1 fichier(s) .parquet
   📂 species=Dog/
      📂 race=wheaten terrier/
         📄 1 fichier(s) .parquet
      📂 race=shiba inu/
     

In [ ]:
# ============================================
# PARTIE 12 : Relecture et Partition Pruning
# ============================================

print("\n🔍 PARTIE 12 : Partition Pruning en Action")
print("="*70)

print("\n1️⃣ Lecture de TOUTES les partitions")

result1 = monitor.execute_and_monitor(
    lambda: spark.read.parquet(output_path).count(),
    "JOB 19: Lecture complète"
)

print(f"✓ {result1:,} images lues")

print("\n2️⃣ Lecture d'UNE SEULE espèce (partition pruning)")

result2 = monitor.execute_and_monitor(
    lambda: spark.read.parquet(output_path)
        .filter(F.col("species") == "Cat")
        .count(),
    "JOB 20: Lecture chats uniquement"
)

print(f"✓ {result2:,} images de chats")
print("💡 Spark a lu SEULEMENT le dossier species=Cat/")

print("\n3️⃣ Lecture d'UNE race spécifique")

result3 = monitor.execute_and_monitor(
    lambda: spark.read.parquet(output_path)
        .filter((F.col("species") == "Cat") & (F.col("race") == "Abyssinian"))
        .select("filename", "width", "height")
        .show(5),
    "JOB 21: Lecture Abyssinian uniquement"
)

print("💡 Spark a lu SEULEMENT species=Cat/race=Abyssinian/")

# Afficher le plan d'exécution
print("\n📋 Plan d'exécution (vérifier le partition pruning):")
spark.read.parquet(output_path) \
    .filter(F.col("species") == "Cat") \
    .explain()

print("\n🔍 Cherchez 'PartitionFilters: [isnotnull(species), (species = Cat)]'")


🔍 PARTIE 12 : Partition Pruning en Action

1️⃣ Lecture de TOUTES les partitions

🔵 JOB 19: Lecture complète
📌 Tracking ID: 9ca80ee4

✅ SUCCESS
⏱️  Durée: 1.84s
📊 Spark Job ID(s): [37, 36, 35, 34]
✓ 1,000 images lues

2️⃣ Lecture d'UNE SEULE espèce (partition pruning)

🔵 JOB 20: Lecture chats uniquement
📌 Tracking ID: b76f0af1

✅ SUCCESS
⏱️  Durée: 1.26s
📊 Spark Job ID(s): [41, 40, 39, 38]
✓ 66 images de chats
💡 Spark a lu SEULEMENT le dossier species=Cat/

3️⃣ Lecture d'UNE race spécifique

🔵 JOB 21: Lecture Abyssinian uniquement
📌 Tracking ID: a0a8dfe1
+--------+-----+------+
|filename|width|height|
+--------+-----+------+
+--------+-----+------+


✅ SUCCESS
⏱️  Durée: 0.95s
📊 Spark Job ID(s): [43, 42]
💡 Spark a lu SEULEMENT species=Cat/race=Abyssinian/

📋 Plan d'exécution (vérifier le partition pruning):
== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [filename#3115,width#3116,height#3117,channels#3118,size_mb#3119,content#3120,species#3121,race#3122] Batched: true, DataF

In [ ]:
# ============================================
# PARTIE 13 : Filtrage et Sélection Avancée
# ============================================

print("\n🔎 PARTIE 13 : Filtrage et Sélection")
print("="*70)

print("\n1️⃣ Images avec résolution minimale (> 300x300)")

result = monitor.execute_and_monitor(
    lambda: df_with_dims.filter((F.col("width") >= 300) & (F.col("height") >= 300))
        .groupBy("species")
        .agg(F.count("*").alias("nb_images"))
        .show(),
    "JOB 22: Filtrer résolution minimale"
)

print("\n2️⃣ Images presque carrées (aspect ratio entre 0.9 et 1.1)")

result = monitor.execute_and_monitor(
    lambda: df_with_dims.filter((F.col("aspect_ratio") >= 0.9) & (F.col("aspect_ratio") <= 1.1))
        .select("filename", "width", "height", "aspect_ratio")
        .show(10),
    "JOB 23: Images carrées"
)

print("\n3️⃣ Échantillonnage aléatoire stratifié (10 images par race)")

result = monitor.execute_and_monitor(
    lambda: df_with_dims.groupBy("race")
        .agg(F.collect_list("filename").alias("filenames"))
        .withColumn("sample", F.slice(F.col("filenames"), 1, 10))
        .select("race", F.size("sample").alias("nb_echantillon"))
        .show(10),
    "JOB 24: Échantillonnage par race"
)


🔎 PARTIE 13 : Filtrage et Sélection

1️⃣ Images avec résolution minimale (> 300x300)

🔵 JOB 22: Filtrer résolution minimale
📌 Tracking ID: 9ee62a54
+-------+---------+
|species|nb_images|
+-------+---------+
|    Cat|      632|
|    Dog|     5688|
+-------+---------+


✅ SUCCESS
⏱️  Durée: 7.67s
📊 Spark Job ID(s): [47, 46]

2️⃣ Images presque carrées (aspect ratio entre 0.9 et 1.1)

🔵 JOB 23: Images carrées
📌 Tracking ID: d2df9402
+--------------------+-----+------+------------+
|            filename|width|height|aspect_ratio|
+--------------------+-----+------+------------+
|  Abyssinian_178.jpg|  400|   400|        1.00|
|german_shorthaire...|  500|   500|        1.00|
|english_cocker_sp...|  500|   500|        1.00|
|  leonberger_104.jpg|  500|   460|        1.09|
|     keeshond_61.jpg|  500|   465|        1.08|
|newfoundland_180.jpg|  500|   500|        1.00|
|newfoundland_190.jpg|  500|   500|        1.00|
|newfoundland_129.jpg|  500|   500|        1.00|
|    shiba_inu_84.jpg|  5

In [ ]:
# ============================================
# PARTIE 14 : Comparaison Formats de Stockage
# ============================================

print("\n⚖️ PARTIE 14 : Parquet vs Fichiers Bruts")
print("="*70)

print("""
💡 COMPARAISON
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Fichiers JPG bruts :
  ✓ Format image standard
  ✓ Compression optimale pour images
  ✗ Pas de métadonnées structurées
  ✗ Pas de partition pruning
  ✗ Lecture séquentielle uniquement

Parquet avec images binaires :
  ✓ Métadonnées + images dans un seul format
  ✓ Partitionnement (lecture sélective)
  ✓ Schéma structuré
  ✓ Compression columnar
  ✗ Légèrement plus gros (métadonnées)

Recommandation :
  • Images brutes : stockage long terme, partage
  • Parquet : pipelines Spark, analytics, features
""")

# Comparer les tailles
import subprocess

original_size = subprocess.check_output(['du', '-sh', images_path]).split()[0].decode('utf-8')
parquet_size = subprocess.check_output(['du', '-sh', output_path]).split()[0].decode('utf-8')

print(f"\n💾 Comparaison des tailles (1000 images) :")
print(f"   Fichiers JPG originaux : ~{original_size} (pour toutes les images)")
print(f"   Parquet avec métadonnées : {parquet_size}")


⚖️ PARTIE 14 : Parquet vs Fichiers Bruts

💡 COMPARAISON
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Fichiers JPG bruts :
  ✓ Format image standard
  ✓ Compression optimale pour images
  ✗ Pas de métadonnées structurées
  ✗ Pas de partition pruning
  ✗ Lecture séquentielle uniquement

Parquet avec images binaires :
  ✓ Métadonnées + images dans un seul format
  ✓ Partitionnement (lecture sélective)
  ✓ Schéma structuré
  ✓ Compression columnar
  ✗ Légèrement plus gros (métadonnées)

Recommandation :
  • Images brutes : stockage long terme, partage
  • Parquet : pipelines Spark, analytics, features


💾 Comparaison des tailles (1000 images) :
   Fichiers JPG originaux : ~775M (pour toutes les images)
   Parquet avec métadonnées : 192M


In [ ]:
# ============================================
# PARTIE 15 : Analyse Exploratoire Finale
# ============================================

print("\n📊 PARTIE 15 : Analyse Exploratoire Finale")
print("="*70)

print("\n1️⃣ Images les plus grandes et les plus petites")

result = monitor.execute_and_monitor(
    lambda: df_with_dims.orderBy(F.col("pixels").desc())
        .select("filename", "race", "width", "height", "pixels")
        .show(5),
    "JOB 25: Plus grandes images"
)

print("\n2️⃣ Races avec images moyennes les plus grandes")

result = monitor.execute_and_monitor(
    lambda: df_with_dims.groupBy("race", "species")
        .agg(
            F.avg("pixels").alias("pixels_moyen"),
            F.count("*").alias("nb_images")
        )
        .orderBy(F.col("pixels_moyen").desc())
        .limit(10)
        .show(truncate=False),
    "JOB 26: Races avec plus grandes images"
)

print("\n3️⃣ Distribution de la luminosité par espèce")

# Créer le DataFrame avec features
df_with_features = df_sample.withColumn(
    "color_features",
    extract_color_features(F.col("content"))
).withColumn(
    "brightness",
    F.col("color_features.brightness")
).drop("color_features")

result = monitor.execute_and_monitor(
    lambda: df_with_features.groupBy("species")
        .agg(
            F.avg("brightness").alias("luminosite_moyenne"),
            F.min("brightness").alias("luminosite_min"),
            F.max("brightness").alias("luminosite_max")
        )
        .show(),
    "JOB 27: Luminosité par espèce"
)


📊 PARTIE 15 : Analyse Exploratoire Finale

1️⃣ Images les plus grandes et les plus petites

🔵 JOB 25: Plus grandes images
📌 Tracking ID: d10aece9
+--------------------+------------+-----+------+-------+
|            filename|        race|width|height| pixels|
+--------------------+------------+-----+------+-------+
|Egyptian_Mau_162.jpg|Egyptian Mau| 3264|  2448|7990272|
|Egyptian_Mau_165.jpg|Egyptian Mau| 2560|  1920|4915200|
|Egyptian_Mau_196.jpg|Egyptian Mau| 1886|  2606|4914916|
| Egyptian_Mau_20.jpg|Egyptian Mau| 1440|  2160|3110400|
|Egyptian_Mau_182.jpg|Egyptian Mau| 1999|  1499|2996501|
+--------------------+------------+-----+------+-------+
only showing top 5 rows

✅ SUCCESS
⏱️  Durée: 3.05s
📊 Spark Job ID(s): [52]

2️⃣ Races avec images moyennes les plus grandes

🔵 JOB 26: Races avec plus grandes images
📌 Tracking ID: 3abac3dd
+----------------+-------+------------------+---------+
|race            |species|pixels_moyen      |nb_images|
+----------------+-------+-----------